# Import libraries

In [1]:
#arcgis
!pip install arcgis
from copy import deepcopy
from datetime import datetime
from IPython.display import HTML
from arcgis.gis import GIS
import arcgis.network as network
import arcgis.geocoding as geocoding

import pandas as pd
import cx_Oracle
from sqlalchemy import create_engine, types
pd.set_option('display.max_columns', None)
from scipy import stats
import json
import re

     |████████████████████████████████| 2.0 MB 962 kB/s eta 0:00:01
  Using cached pandas-1.1.2-cp36-cp36m-manylinux1_x86_64.whl (10.5 MB)
  Using cached keyring-21.4.0-py3-none-any.whl (31 kB)
Processing /tmp/wsuser/.cache/pip/wheels/e3/ff/dc/6533fc2f27e57e80c482fa4f99872b91b8395d74001dc259a0/lerc-0.1.0-py3-none-any.whl
  Using cached jupyterlab-2.2.8-py3-none-any.whl (7.8 MB)
  Using cached requests_oauthlib-1.3.0-py2.py3-none-any.whl (23 kB)
  Using cached requests_toolbelt-0.9.1-py2.py3-none-any.whl (54 kB)
  Using cached requests_ntlm-1.1.0-py2.py3-none-any.whl (5.7 kB)
  Using cached jeepney-0.4.3-py3-none-any.whl (21 kB)
  Using cached jupyterlab_server-1.2.0-py3-none-any.whl (29 kB)
  Using cached oauthlib-3.1.0-py2.py3-none-any.whl (147 kB)
  Using cached ntlm_auth-1.5.0-py2.py3-none-any.whl (29 kB)


  Using cached json5-0.9.5-py2.py3-none-any.whl (17 kB)
  Created wheel for arcgis: filename=arcgis-1.8.2-py2.py3-none-any.whl size=2622663 sha256=389d3fa3e09bf92ff0cffc9169046bc2291b56f72e3a00a20826fc124a30ec72
  Stored in directory: /tmp/wsuser/.cache/pip/wheels/8b/11/0c/21329dd08f13936a0b31b0cd27cbe712348eb50e8133ca4923
Successfully built arcgis
ERROR: hdijupyterutils 0.12.9 requires jupyter>=1, which is not installed.
ERROR: brunel 2.3 requires JPype1-py3, which is not installed.
ERROR: watson-machine-learning-client-v4 1.0.95 has requirement ibm-cos-sdk==2.6.0, but you'll have ibm-cos-sdk 2.5.1 which is incompatible.
ERROR: watson-machine-learning-client-v4 1.0.95 has requirement pandas<=0.25.3, but you'll have pandas 1.1.2 which is incompatible.
ERROR: jupyterlab-server 1.2.0 has requirement jsonschema>=3.0.1, but you'll have jsonschema 2.6.0 which is incompatible.
  Attempting uninstall: pandas
    Found existing installation: pandas 0.24.2
    Uninstalling pandas-0.24.2:
      

# Get Credentials

In [2]:
from project_lib import Project
project = Project.access()
uat_credentials = project.get_connection(name="alex_uat")
arcgis_credentials = project.get_connection(name="Arcgis")

Missing personal credentials for "Arcgis" (40c83ad4-cfdf-4616-acab-2a1df1faf29d).
Please configure them in the Watson Studio project.


# Setup DB Connection

In [3]:
host = uat_credentials['host']
port = uat_credentials['port']
user = uat_credentials['username']
password = uat_credentials['password']
service_name = uat_credentials['service_name']

sid = cx_Oracle.makedsn(host = host, 
                        port = port, 
                        service_name = service_name)
 
cstr = 'oracle://{user}:{password}@{sid}'.format(
    user=user,
    password=password,
    sid=sid
)

engine =  create_engine(
    cstr,
    convert_unicode=False,
    pool_recycle=10,
    pool_size=50,
    echo=True
)

# Read in Dataset with lat/lon

In [4]:
PROV_CLNT_ADDR = pd.read_sql('select * from PROV_CLNT_ADDR where rownum < 100', engine)

PROV_CLNT_ADDR.head()

2020-09-29 21:40:08,305 INFO sqlalchemy.engine.base.Engine SELECT USER FROM DUAL
2020-09-29 21:40:08,307 INFO sqlalchemy.engine.base.Engine {}
2020-09-29 21:40:08,355 INFO sqlalchemy.engine.base.Engine SELECT CAST('test plain returns' AS VARCHAR(60 CHAR)) AS anon_1 FROM DUAL
2020-09-29 21:40:08,357 INFO sqlalchemy.engine.base.Engine {}
2020-09-29 21:40:08,400 INFO sqlalchemy.engine.base.Engine SELECT CAST('test unicode returns' AS NVARCHAR2(60)) AS anon_1 FROM DUAL
2020-09-29 21:40:08,402 INFO sqlalchemy.engine.base.Engine {}
2020-09-29 21:40:08,486 INFO sqlalchemy.engine.base.Engine select value from nls_session_parameters where parameter = 'NLS_NUMERIC_CHARACTERS'
2020-09-29 21:40:08,488 INFO sqlalchemy.engine.base.Engine {}
2020-09-29 21:40:08,533 INFO sqlalchemy.engine.base.Engine SELECT table_name FROM all_tables WHERE table_name = :name AND owner = :schema_name
2020-09-29 21:40:08,535 INFO sqlalchemy.engine.base.Engine {'name': 'select * from PROV_CLNT_ADDR where rownum < 100', '

,clm_lne_fact_sk,prscrb_prov_loc_id,prscrb_prov_loc_dim_sk,mcaid_id,clnt_dim_sk,prov_loc_id,svc_addr_lne_1_tx,svc_addr_lne_2_tx,svc_addr_lne_3_tx,svc_cty_nm,svc_st_cd,svc_pstl_cd,svc_zip_pls_4_cd,svc_addr_lat_nbr,svc_addr_long_nbr,mcaid_id_2,home_addr_lne_1_tx,home_addr_lne_2_tx,home_addr_lne_3_tx,home_cty_nm,home_st_cd,home_pstl_cd,home_zip_pls_4_cd,home_addr_lat_nbr,home_addr_long_nbr
0,273466776,121614,517046,G911282,17630605,121614,3555 LUTHERAN PKWY,STE 200,None,WHEAT RIDGE,CO,80033,6027,39.766269,-105.08849,G911282,155 S GARLAND ST,None,None,LAKEWOOD,CO,80226,1036,39.714067,-105.101625
1,247457971,165691,450888,P390376,15729856,165691,6041 S SYRACUSE WAY,SUITE 220,None,GREENWOOD VILLAGE,CO,80111,4716,39.604873,-104.89947,P390376,8330 E QUINCY AVE,APT B201,None,DENVER,CO,80237,2445,39.638670,-104.892660
2,274440217,165691,450888,P390376,16247731,165691,6041 S SYRACUSE WAY,SUITE 220,None,GREENWOOD VILLAGE,CO,80111,4716,39.604873,-104.89947,P390376,8330 E QUINCY AVE,APT B201,None,DENVER,CO,80237,2445,39.638670,-104.892660
3,258871615,165691,450888,P390376,16247731,165691,6041 S SYRACUSE WAY,SUITE 220,None,GREENWOOD VILLAGE,CO,80111,4716,39.604873,-104.89947,P390376,8330 E QUINCY AVE,APT B201,None,DENVER,CO,80237,2445,39.638670,-104.892660
4,258560234,103453,497463,S530146,17293014,103453,2525 4TH ST,STE 202,None,BOULDER,CO,80304,4014,40.022225,-105.29149,S530146,1409 E 19TH ST,APT C,None,PUEBLO,CO,81001,2664,38.285035,-104.587016


# Test ARCGIS API

In [7]:
my_gis = GIS('https://www.arcgis.com', arcgis_credentials['username'], arcgis_credentials['password'])

route_service_url = my_gis.properties.helperServices.route.url
route_service = network.RouteLayer(route_service_url, gis=my_gis)
route_layer = network.RouteLayer(route_service_url, gis=my_gis)
result = route_layer.solve(stops='''18.068598,59.329268; 18.068598,59.429268''',
                           return_directions=False, return_routes=True, 
                           output_lines='esriNAOutputLineNone',
                           return_barriers=False, return_polygon_barriers=False, 
                           return_polyline_barriers=False)

travel_time = result['routes']['features'][0]['attributes']['Total_TravelTime']

print(travel_time)

18.532682893163145


# Function to compute distance

In [8]:
def get_driving_distance(lat1,lon1,lat2,lon2):
    stops1 = '''{},{}; {},{}'''.format(lon1,lat1,lon2,lat2)
    result = route_layer.solve(stops=stops1,
                               return_directions=False, return_routes=True, 
                               output_lines='esriNAOutputLineNone',
                               return_barriers=False, return_polygon_barriers=False, 
                               return_polyline_barriers=False)
    
    travel_time = result['routes']['features'][0]['attributes']['Total_TravelTime']
    total_miles = result['routes']['features'][0]['attributes']['Total_Miles']
    return(total_miles)

In [9]:
driving_dist = PROV_CLNT_ADDR.apply(lambda x: get_driving_distance(x['home_addr_lat_nbr'], x['home_addr_long_nbr'],
                                              x['svc_addr_lat_nbr'], x['svc_addr_long_nbr']), axis=1)

In [11]:
PROV_CLNT_ADDR['driving_dist'] = pd.Series(driving_dist)

In [12]:
PROV_CLNT_ADDR

,clm_lne_fact_sk,prscrb_prov_loc_id,prscrb_prov_loc_dim_sk,mcaid_id,clnt_dim_sk,prov_loc_id,svc_addr_lne_1_tx,svc_addr_lne_2_tx,svc_addr_lne_3_tx,svc_cty_nm,svc_st_cd,svc_pstl_cd,svc_zip_pls_4_cd,svc_addr_lat_nbr,svc_addr_long_nbr,mcaid_id_2,home_addr_lne_1_tx,home_addr_lne_2_tx,home_addr_lne_3_tx,home_cty_nm,home_st_cd,home_pstl_cd,home_zip_pls_4_cd,home_addr_lat_nbr,home_addr_long_nbr,driving_dist
0,273466776,121614,517046,G911282,17630605,121614,3555 LUTHERAN PKWY,STE 200,None,WHEAT RIDGE,CO,80033,6027,39.766269,-105.088490,G911282,155 S GARLAND ST,None,None,LAKEWOOD,CO,80226,1036,39.714067,-105.101625,5.230173
1,247457971,165691,450888,P390376,15729856,165691,6041 S SYRACUSE WAY,SUITE 220,None,GREENWOOD VILLAGE,CO,80111,4716,39.604873,-104.899470,P390376,8330 E QUINCY AVE,APT B201,None,DENVER,CO,80237,2445,39.638670,-104.892660,3.287969
2,274440217,165691,450888,P390376,16247731,165691,6041 S SYRACUSE WAY,SUITE 220,None,GREENWOOD VILLAGE,CO,80111,4716,39.604873,-104.899470,P390376,8330 E QUINCY AVE,APT B201,None,DENVER,CO,80237,2445,39.638670,-104.892660,3.287969
3,258871615,165691,450888,P390376,16247731,165691,6041 S SYRACUSE WAY,SUITE 220,None,GREENWOOD VILLAGE,CO,80111,4716,39.604873,-104.899470,P390376,8330 E QUINCY AVE,APT B201,None,DENVER,CO,80237,2445,39.638670,-104.892660,3.287969
4,258560234,103453,497463,S530146,17293014,103453,2525 4TH ST,STE 202,None,BOULDER,CO,80304,4014,40.022225,-105.291490,S530146,1409 E 19TH ST,APT C,None,PUEBLO,CO,81001,2664,38.285035,-104.587016,140.789280
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,247542641,104310,544948,Y300908,19017251,104310,2950 E HARMONY RD,STE 190,None,FT COLLINS,CO,80528,3430,40.523300,-105.023611,Y300908,9499 W 56TH PL,APT 5,None,ARVADA,CO,80002,2167,39.799021,-105.103675,58.113596
95,267216649,127373,534497,G635383,19561856,127373,8030 LEE DR,None,None,ARVADA,CO,80005,2078,39.842632,-105.112049,G635383,8277 TELLER CT,None,None,ARVADA,CO,80003,1621,39.847591,-105.077222,2.291150
96,258946526,127373,534497,G635383,19561856,127373,8030 LEE DR,None,None,ARVADA,CO,80005,2078,39.842632,-105.112049,G635383,8277 TELLER CT,None,None,ARVADA,CO,80003,1621,39.847591,-105.077222,2.291150
97,262181607,127373,534497,O247439,18018002,127373,8030 LEE DR,None,None,ARVADA,CO,80005,2078,39.842632,-105.112049,O247439,12187 W 69TH AVE,None,None,ARVADA,CO,80004,2322,39.822480,-105.135768,2.878217
